
# IT25101600 – Logistic Regression
## Fitness Classification Project

This notebook contains the complete individual implementation for **Logistic Regression**.

### Included
- Dataset loading
- Categorical cleaning
- BMI feature engineering
- Train/test split
- Median imputation
- Scaling where required
- GridSearchCV hyperparameter tuning
- 5-fold stratified cross-validation
- Accuracy, Precision, Recall, F1-score, ROC-AUC
- Confusion matrix
- Classification report
- Best parameter selection
- Viva-ready explanations


In [ ]:

# 1. IMPORT LIBRARIES

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    ConfusionMatrixDisplay
)

import joblib

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

print("Libraries imported successfully.")


In [ ]:

# 2. LOAD DATASET

candidate_paths = [
    "fitness_dataset.csv",
    "/content/fitness_dataset.csv",
    "/mnt/data/project_unpacked/fitness_dataset.csv"
]

data_path = None

for p in candidate_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    try:
        from google.colab import files
        print("Please upload fitness_dataset.csv")
        uploaded = files.upload()
        data_path = next(iter(uploaded.keys()))
    except Exception as e:
        raise FileNotFoundError(
            "fitness_dataset.csv was not found. "
            "Upload it in Google Colab or place it beside the notebook."
        ) from e

df = pd.read_csv(data_path)

print("Loaded:", data_path)
print("Shape:", df.shape)
display(df.head())



## 3. Preprocessing

The dataset is prepared by:
- converting `smokes` from yes/no/1/0 into 1/0
- encoding `gender`
- creating BMI
- keeping missing-value handling inside the model pipeline


In [ ]:

# 3. CLEAN DATA + FEATURE ENGINEERING

data = df.copy()

data["smokes"] = (
    data["smokes"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"yes": "1", "no": "0"})
)
data["smokes"] = pd.to_numeric(data["smokes"], errors="coerce")

data["gender"] = (
    data["gender"]
    .astype(str)
    .str.strip()
    .str.upper()
    .map({"F": 0, "M": 1})
)

data["bmi"] = data["weight_kg"] / ((data["height_cm"] / 100) ** 2)

print("Missing values:")
display(data.isnull().sum().to_frame("Missing_Count"))

print("\nTarget distribution:")
display(
    pd.DataFrame({
        "Count": data["is_fit"].value_counts().sort_index(),
        "Percentage": (
            data["is_fit"]
            .value_counts(normalize=True)
            .sort_index()
            .mul(100)
            .round(2)
        )
    })
)


In [ ]:

# 4. DEFINE FEATURES AND TARGET

X = data.drop(columns=["is_fit"])
y = data["is_fit"].astype(int)

print("Features:")
print(X.columns.tolist())

print("\nTarget: is_fit")


In [ ]:

# 5. TRAIN / TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("Number of features:", X_train.shape[1])


# 6. Build and tune Logistic Regression

In [ ]:

# 6. MODEL PIPELINE

model_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ))
])

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

grid = GridSearchCV(
    estimator=model_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

grid.fit(X_train, y_train)

print("Best Parameters:")
print(grid.best_params_)

print("\nBest Cross-Validation F1:")
print(round(grid.best_score_, 4))


# 7. Evaluate the tuned model

In [ ]:

# 7. TEST SET EVALUATION

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

if hasattr(best_model, "predict_proba"):
    y_score = best_model.predict_proba(X_test)[:, 1]
elif hasattr(best_model, "decision_function"):
    y_score = best_model.decision_function(X_test)
else:
    y_score = y_pred

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_score)

results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "Best CV F1"
    ],
    "Score": [
        accuracy,
        precision,
        recall,
        f1,
        roc_auc,
        grid.best_score_
    ]
})

display(results.round(4))


In [ ]:

# 8. CLASSIFICATION REPORT

print("Classification Report - Logistic Regression")
print(classification_report(
    y_test,
    y_pred,
    digits=4
))


In [ ]:

# 9. CONFUSION MATRIX

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred
)

plt.title("Confusion Matrix - Logistic Regression")
plt.tight_layout()
plt.show()


# 10. Show tuning variants

In [ ]:

# 10. TOP HYPERPARAMETER VARIANTS

cv_results = pd.DataFrame(grid.cv_results_)

display(
    cv_results[
        [
            "params",
            "mean_train_score",
            "mean_test_score",
            "std_test_score",
            "rank_test_score"
        ]
    ]
    .sort_values("rank_test_score")
    .head(10)
)


# 11. Save model and results

In [ ]:

# 11. SAVE OUTPUTS

OUTPUT_DIR = Path("logistic_regression_results")
OUTPUT_DIR.mkdir(exist_ok=True)

joblib.dump(
    best_model,
    OUTPUT_DIR / "logistic_regression_best_model.joblib"
)

results.to_csv(
    OUTPUT_DIR / "logistic_regression_metrics.csv",
    index=False
)

prediction_df = pd.DataFrame({
    "actual_is_fit": y_test.values,
    "predicted_is_fit": y_pred
})

prediction_df.to_csv(
    OUTPUT_DIR / "logistic_regression_predictions.csv",
    index=False
)

print("Saved files:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print("-", p)



# 12. Viva Notes – Logistic Regression

### Why use this model?
Explain how Logistic Regression works and why it is suitable for binary classification.

### What is GridSearchCV?
GridSearchCV systematically tests multiple hyperparameter combinations using cross-validation.

### Why use 5-fold Stratified Cross Validation?
It evaluates the model across several folds while preserving approximately the same class ratio in each fold.

### Why use F1-score for tuning?
F1 balances Precision and Recall and is useful when the target classes are not perfectly balanced.

### Accuracy
The proportion of all predictions that were correct.

### Precision
Of all observations predicted as fit, how many were actually fit?

### Recall
Of all actually fit observations, how many were identified correctly?

### F1-score
The harmonic mean of Precision and Recall.

### ROC-AUC
Measures the model's ability to separate the positive and negative classes across thresholds.

### What is overfitting?
The model performs very well on training data but does not generalize well to unseen data.

### What is underfitting?
The model is too simple to capture the important patterns in the dataset.
